# 🚀 Laya Decision Engine - Kaggle Server with Ngrok (Multilingual Only)

Notebook tối giản để chạy **Laya Server** trên Kaggle (GPU T4/P100) và public qua **ngrok** với static domain.
Cài đặt trực tiếp từ fork repository [binhminhanh1235/laya-s1-decision](https://github.com/binhminhanh1235/laya-s1-decision) (đã tích hợp toàn bộ các bản vá Tier 1/2 và tối ưu VRAM).

### ⚠️ Yêu cầu trước khi chạy:
1. **Bật Internet**: Tại thanh bên phải Notebook  ->  -> chọn **Internet On**.
2. **Thêm Secret**: Menu trên cùng ->  ->  -> thêm secret với Label  và Value là auth token Ngrok của bạn.

In [ ]:
# =============================================================================
# 1 CELL DUY NHẤT: CÀI ĐẶT, KẾT NỐI NGROK & KHỞI CHẠY LAYA SERVER (MULTILINGUAL)
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

# 1. Cài đặt pyngrok và laya kèm serve & mcp
!pip install -q pyngrok "git+https://github.com/binhminhanh1235/laya-s1-decision.git#egg=laya[serve,mcp]"

from pyngrok import ngrok

# 2. Lấy Ngrok Auth Token từ Kaggle Secrets & cấu hình
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# Đóng tunnel cũ (nếu có khi rerun lại cell) để tránh xung đột domain tĩnh
ngrok.kill()

# 3. Khởi tạo ngrok tunnel với domain cố định
public_url = ngrok.connect(8000, domain="ferret-above-formerly.ngrok-free.app")
print(f"\n{'='*55}")
print(f"👉 Laya Server Domain: {public_url}")
print(f"👉 Health Check URL:  {public_url}/health")
print(f"👉 Inference API:      {public_url}/v1/systemone")
print(f"👉 Active Checkpoint:  multilingual only")
print(f"{'='*55}\n")

# 4. Cấu hình CHỈ tải và chạy duy nhất checkpoint 'multilingual'
os.environ["LAYA_MODELS"] = "multilingual"
os.environ["LAYA_MAX_LOADED"] = "1"                # Giới hạn giữ tối đa 1 model trong VRAM

# 5. Chạy Laya server trên port 8000 (tự động nhận GPU CUDA nếu bật Accelerator)
!python -m laya.serve